# landing_ingestion: the recurring pull to the landing volume

Pulls the last `DAYS_BACK` days (default 3) for every endpoint and area and saves the API responses,
exactly as received, in the landing volume. This is the notebook the job runs every 6 hours (later, daily).

One file per API call: `<volume>/<endpoint>/ingest_date=2026-11-07/20261107T093015Z_landing_AVA_w000.json`

* If one pull fails, the notebook stops and the job shows as failed. The next run looks back 3 days,
  so it picks up what was missed.
* `initial_ingestion` holds the same helper code. If you change the config or a function, change both.

In [0]:
import json
import os
import time
from datetime import datetime, timedelta, timezone

import requests

In [0]:
# ---------------- CONFIG: edit these ----------------
CATALOG = "dbw_energy_analytics_dev"
SCHEMA = "raw"
VOLUME = "raw_files"

SECRET_SCOPE = "energy-secrets"
API_KEY_SECRET_NAME = "energyg-api-key"     # must match the secret name in Key Vault

REGIONS = ["AVA", "BPAT", "CISO", "PGE", "SCL"]
ENDPOINTS = ["region-data", "fuel-type-data"]
# ----------------------------------------------------

BASE_URL = "https://api.eia.gov/v2/electricity/rto"
HOUR_FMT = "%Y-%m-%dT%H"
MAX_RETRIES = 4
RETRY_WAIT_SECONDS = [5, 15, 30]       # wait before attempt 2, 3 and 4
PAUSE_SECONDS = 1                      # pause between calls

LANDING_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
if not os.path.isdir(LANDING_ROOT):
    raise RuntimeError(f"Volume path not found: {LANDING_ROOT}. Check the names in the config cell.")

In [0]:
def log(level, message, **kwargs):
    print(json.dumps({
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "level": level,
        "message": message,
        **kwargs,
    }))


def get_api_key():
    try:
        return dbutils.secrets.get(scope=SECRET_SCOPE, key=API_KEY_SECRET_NAME)
    except Exception as e:
        log("ERROR", "Failed to retrieve API key from Key Vault", error=str(e))
        raise


def fetch(endpoint, region, start, end, api_key):
    """One API call with retries. Returns the response."""
    url = f"{BASE_URL}/{endpoint}/data/"
    params = {
        "api_key": api_key,
        "frequency": "hourly",
        "data[0]": "value",
        "facets[respondent][]": region,
        "start": start,
        "end": end,
        "sort[0][column]": "period",
        "sort[0][direction]": "asc",
        "length": 5000,                  # the most the API returns in one call
    }
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.get(url, params=params, timeout=60)
            if response.status_code == 200:
                return response
            log("WARNING", "API returned an error status", endpoint=endpoint, region=region,
                status=response.status_code, attempt=attempt)
            if response.status_code < 500 and response.status_code != 429:
                break                    # bad request or bad key: retrying will not help
        except requests.exceptions.Timeout:
            log("WARNING", "Request timed out", endpoint=endpoint, region=region, attempt=attempt)
        except requests.exceptions.RequestException as e:
            # log only the error type: the full message can contain the URL with the API key
            log("WARNING", "Request failed", endpoint=endpoint, region=region,
                error_type=type(e).__name__, attempt=attempt)
        if attempt < MAX_RETRIES:
            time.sleep(RETRY_WAIT_SECONDS[attempt - 1])
    raise RuntimeError(f"Could not fetch {endpoint} {region} {start} -> {end}")


def pull_and_save(endpoint, region, start, end, window_no, api_key, mode):
    """Fetches one endpoint, area and date range, and saves the response exactly as received."""
    response = fetch(endpoint, region, start, end, api_key)
    data = response.json()
    if "response" not in data:
        raise RuntimeError(f"Unexpected response for {endpoint} {region}")

    total = int(data["response"]["total"])          # the API returns total as text
    rows = len(data["response"]["data"])
    if rows < total:
        # the API only returns 5,000 rows per call: never save an incomplete pull silently
        raise RuntimeError(f"{endpoint} {region}: got {rows} of {total} rows, use a smaller date range")
    if api_key in response.text:
        raise RuntimeError(f"{endpoint} {region}: API key found in the response, not saving it")
    if rows == 0:
        log("WARNING", "API returned zero rows, nothing saved", endpoint=endpoint, region=region,
            start=start, end=end)
        return 0

    now = datetime.now(timezone.utc)
    folder = f"{LANDING_ROOT}/{endpoint}/ingest_date={now.strftime('%Y-%m-%d')}"
    os.makedirs(folder, exist_ok=True)
    path = f"{folder}/{now.strftime('%Y%m%dT%H%M%SZ')}_{mode}_{region}_w{window_no:03d}.json"
    with open(path, "w") as f:
        f.write(response.text)               # raw response, nothing added or removed
    log("INFO", "Saved", endpoint=endpoint, region=region, start=start, end=end, rows=rows, file=path)
    return rows

In [0]:
DAYS_BACK = 3      # how far back each run looks, to catch data that arrives late

now_hour = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
windows = [((now_hour - timedelta(days=DAYS_BACK)).strftime(HOUR_FMT), now_hour.strftime(HOUR_FMT))]

In [0]:
api_key = get_api_key()
log("INFO", "Run started", mode="landing", windows=len(windows), regions=REGIONS, endpoints=ENDPOINTS)

files_saved, rows_saved = 0, 0
for endpoint in ENDPOINTS:
    for region in REGIONS:
        for window_no, (start, end) in enumerate(windows):
            rows = pull_and_save(endpoint, region, start, end, window_no, api_key, "landing")
            if rows:
                files_saved += 1
                rows_saved += rows
            time.sleep(PAUSE_SECONDS)

log("INFO", "Run finished", mode="landing", files_saved=files_saved, rows_saved=rows_saved)

{"timestamp": "2026-11-18T07:21:18.931511+00:00", "level": "INFO", "message": "Run started", "mode": "landing", "windows": 1, "regions": ["AVA", "BPAT", "CISO", "PGE", "SCL"], "endpoints": ["region-data", "fuel-type-data"]}
{"timestamp": "2026-11-18T07:21:22.338336+00:00", "level": "INFO", "message": "Saved", "endpoint": "region-data", "region": "AVA", "start": "2026-11-15T07", "end": "2026-11-18T07", "rows": 243, "file": "/Volumes/dbw_energy_analytics_dev/raw/raw_files/region-data/ingest_date=2026-11-18/20261118T072120Z_landing_AVA_w000.json"}
{"timestamp": "2026-11-18T07:21:24.961599+00:00", "level": "INFO", "message": "Saved", "endpoint": "region-data", "region": "BPAT", "start": "2026-11-15T07", "end": "2026-11-18T07", "rows": 243, "file": "/Volumes/dbw_energy_analytics_dev/raw/raw_files/region-data/ingest_date=2026-11-18/20261118T072124Z_landing_BPAT_w000.json"}
{"timestamp": "2026-11-18T07:21:27.564273+00:00", "level": "INFO", "message": "Saved", "endpoint": "region-data", "regio